In [12]:
# !pip install gymnasium minigrid imageio protobuf==3.20.1

In [13]:
# import subprocess
# import time

# model_name = "Qwen/Qwen2.5-7B-Instruct"

# print(f"Loading vLLM server with model: {model_name}")
# print("Startup takes about 3–4 minutes...")

# command = f"nohup python -m vllm.entrypoints.openai.api_server \
# --model {model_name} \
# --dtype auto \
# --api-key empty \
# --port 8000 \
# --gpu-memory-utilization 0.9 \
# --enforce-eager \
# > vllm_server.log 2>&1 &"

# subprocess.run(command, shell=True)

Loading vLLM server with model: Qwen/Qwen2.5-7B-Instruct
Startup takes about 3–4 minutes...


CompletedProcess(args='nohup python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-7B-Instruct --dtype auto --api-key empty --port 8000 --gpu-memory-utilization 0.9 --enforce-eager > vllm_server.log 2>&1 &', returncode=0)

In [14]:
# import requests
# import time

# url = "http://localhost:8000/v1/models"
# headers = {"Authorization": "Bearer empty"}

# print("Waiting for vLLM server to start (up to 20 retries)...")
# for i in range(20):
#     try:
#         response = requests.get(url, headers=headers)
#         if response.status_code == 200:
#             print("✅ vLLM loaded successfully")
#             print("Model:", response.json()['data'][0]['id'])
#             break
#     except requests.exceptions.ConnectionError:
#         print(f"[{i+1}/20] Server not ready, retrying...")
#         time.sleep(15)
# else:
#     print("❌ Server failed to start. Check vllm_server.log")

Waiting for vLLM server to start (up to 20 retries)...
[1/20] Server not ready, retrying...
[2/20] Server not ready, retrying...
[3/20] Server not ready, retrying...
[4/20] Server not ready, retrying...
[5/20] Server not ready, retrying...
[6/20] Server not ready, retrying...
[7/20] Server not ready, retrying...
[8/20] Server not ready, retrying...
[9/20] Server not ready, retrying...
[10/20] Server not ready, retrying...
[11/20] Server not ready, retrying...
[12/20] Server not ready, retrying...
[13/20] Server not ready, retrying...
[14/20] Server not ready, retrying...
[15/20] Server not ready, retrying...
[16/20] Server not ready, retrying...
[17/20] Server not ready, retrying...
[18/20] Server not ready, retrying...
[19/20] Server not ready, retrying...
[20/20] Server not ready, retrying...
❌ Server failed to start. Check vllm_server.log


In [ ]:
import os


if os.path.exists("vllm_server.log"):
    with open("vllm_server.log", "r") as f:
        print(f.read())
else:
    print("vllm_server.log not found. Start the server cell first if you want to inspect logs.")

2026-03-10 14:19:54.389030: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773152394.412045    8516 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773152394.419149    8516 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773152394.435935    8516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773152394.435977    8516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773152394.435979    8516 computation_placer.cc:177] computation placer alr

In [ ]:
import gymnasium as gym
from minigrid.wrappers import FullyObsWrapper
from minigrid.core.actions import Actions


class MinigridTextWrapper:
    DIR_VEC = {0: (1, 0), 1: (0, 1), 2: (-1, 0), 3: (0, -1)}
    DIR_NAMES = ["right", "down", "left", "up"]

    def __init__(self, env_id, render_mode=None):
        self.env = gym.make(env_id, render_mode=render_mode)
        self.env = FullyObsWrapper(self.env)
        self.action_map = {
            "turn_left": Actions.left,
            "turn_right": Actions.right,
            "move_forward": Actions.forward,
        }

    def _base(self):
        return self.env.unwrapped

    def _find_goal_pos(self):
        base = self._base()
        grid = base.grid
        for x in range(grid.width):
            for y in range(grid.height):
                cell = grid.get(x, y)
                if cell is not None and getattr(cell, "type", None) == "goal":
                    return (int(x), int(y))
        return None

    def _cell_content(self, grid, x, y):
        if x < 0 or y < 0 or x >= grid.width or y >= grid.height:
            return "wall"
        cell = grid.get(x, y)
        return cell.type if cell else "empty"

    def get_text_obs(self, obs):
        base = self._base()
        grid = base.grid
        ax, ay = int(base.agent_pos[0]), int(base.agent_pos[1])
        agent_dir = int(base.agent_dir)
        facing = self.DIR_NAMES[agent_dir]

        desc = f"Agent is at [{ax}, {ay}] facing {facing}. "

        fdx, fdy = self.DIR_VEC[agent_dir]
        rdx, rdy = self.DIR_VEC[(agent_dir + 1) % 4]
        ldx, ldy = self.DIR_VEC[(agent_dir - 1) % 4]
        bdx, bdy = self.DIR_VEC[(agent_dir + 2) % 4]

        front = self._cell_content(grid, ax + fdx, ay + fdy)
        right = self._cell_content(grid, ax + rdx, ay + rdy)
        left = self._cell_content(grid, ax + ldx, ay + ldy)
        behind = self._cell_content(grid, ax + bdx, ay + bdy)

        desc += f"Nearby - front: {front}, left: {left}, right: {right}, behind: {behind}. "
        desc += f"The cell directly in front of you contains {front}. "

        goal_pos = self._find_goal_pos()
        if goal_pos:
            gx, gy = goal_pos
            dx, dy = gx - ax, gy - ay
            manhattan = abs(dx) + abs(dy)
            desc += f"Goal is at [{gx}, {gy}]. Distance: {manhattan}. "

            dirs_needed = []
            if dx > 0:
                dirs_needed.append("right")
            elif dx < 0:
                dirs_needed.append("left")
            if dy > 0:
                dirs_needed.append("down")
            elif dy < 0:
                dirs_needed.append("up")

            if dirs_needed:
                desc += f"You need to go: {', '.join(dirs_needed)}. "
                if facing in dirs_needed:
                    desc += "You are facing TOWARD the goal. "
                else:
                    desc += "You are NOT facing the goal. "

        lavas = []
        walls = []
        for x in range(grid.width):
            for y in range(grid.height):
                cell = grid.get(x, y)
                if cell is None:
                    continue
                obj_type = getattr(cell, "type", None)
                if obj_type == "lava":
                    lavas.append(f"[{x},{y}]")
                elif obj_type == "wall":
                    walls.append(f"[{x},{y}]")

        if lavas:
            desc += f"Lava: {', '.join(lavas)}. "
        if walls:
            desc += f"Walls: {', '.join(walls)}. "

        return desc

    def reset(self, seed=None):
        if seed is not None:
            obs, _ = self.env.reset(seed=seed)
        else:
            obs, _ = self.env.reset()
        return self.get_text_obs(obs)

    def step(self, action_str):
        action = self.action_map.get(action_str.lower(), Actions.forward)
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        return self.get_text_obs(obs), reward, done, info

In [ ]:
import os


def evaluate_agent(
    agent,
    env_name="MiniGrid-LavaGapS6-v0",
    num_episodes=10,
    max_steps_per_episode=100,
    gif_folder="episode_gifs",
    save_gifs=False,
    seed_base=42,
 ):
    print(f"Evaluating {env_name} for {num_episodes} episodes")

    env = MinigridTextWrapper(env_name, render_mode="rgb_array")
    os.makedirs(gif_folder, exist_ok=True)

    metrics = {
        "success_count": 0,
        "total_steps_success": 0,
        "total_inference_time": 0.0,
        "total_actions": 0,
    }

    for episode in range(num_episodes):
        obs = env.reset(seed=seed_base + episode)
        agent.reset()

        done = False
        step_count = 0
        episode_reward = 0.0
        frames = []
        goal_pos = env._find_goal_pos()

        while not done and step_count < max_steps_per_episode:
            if save_gifs:
                frames.append(env.env.render())

            action, _response, inf_time = agent.act(obs, env.env)

            metrics["total_inference_time"] += inf_time
            metrics["total_actions"] += 1

            obs, reward, done, _ = env.step(action)

            step_count += 1
            episode_reward += reward

        current_pos = tuple(int(v) for v in env.env.unwrapped.agent_pos)
        success = (episode_reward > 0) or (goal_pos is not None and current_pos == goal_pos)

        if success:
            metrics["success_count"] += 1
            metrics["total_steps_success"] += step_count

        if hasattr(agent, "learn"):
            agent.learn(success, max_steps=max_steps_per_episode)

        if save_gifs and frames:
            import imageio

            gif_path = os.path.join(gif_folder, f"{env_name}_episode_{episode + 1}.gif")
            imageio.mimsave(gif_path, frames, fps=5)

        print(f"Episode {episode + 1}/{num_episodes} | Success: {success} | Steps: {step_count}")

    success_rate = metrics["success_count"] / num_episodes * 100
    avg_steps = (
        metrics["total_steps_success"] / metrics["success_count"]
        if metrics["success_count"] > 0 else float("inf")
    )
    avg_inf_time = (
        metrics["total_inference_time"] / metrics["total_actions"]
        if metrics["total_actions"] > 0 else 0.0
    )

    print("\n" + "=" * 40)
    print("Final Metrics")
    print("=" * 40)
    print(f"Success Rate: {success_rate:.2f}%")
    print(f"Avg Steps (successful episodes): {avg_steps:.2f}")
    print(f"Avg Inference Time per step: {avg_inf_time:.4f} sec")
    print("=" * 40)

    return metrics

In [ ]:
import matplotlib.pyplot as plt


def debug_agent(agent, env_name="MiniGrid-LavaGapS6-v0", steps=10):
    env = MinigridTextWrapper(env_name, render_mode="rgb_array")
    obs = env.reset(seed=42)
    agent.reset()

    for step in range(steps):
        print(f"\nSTEP {step + 1}")
        print("\nOBSERVATION:\n", obs)

        action, response, latency = agent.act(obs, env.env)

        print("\nREASONING:\n", response)
        print(f"\nACTION: {action} | latency: {latency:.4f}s")

        frame = env.env.render()
        plt.imshow(frame)
        plt.axis("off")
        plt.title(f"Step {step + 1}: {action}")
        plt.show()

        obs, reward, done, _ = env.step(action)
        if done:
            print(f"\nEpisode finished | reward: {reward}")
            break

    env.env.close()

In [ ]:
import re
from dataclasses import dataclass


@dataclass
class ThoughtTemplate:
    name: str
    description: str
    reasoning_pattern: str
    usage_count: int = 0
    success_rate: float = 0.0

    def instantiate(self, distilled_obs):
        context = []
        if distilled_obs.get("agent_pos") is not None:
            context.append(f"agent={distilled_obs['agent_pos']}")
        if distilled_obs.get("facing") is not None:
            context.append(f"facing={distilled_obs['facing']}")
        if distilled_obs.get("target_pos") is not None:
            context.append(f"goal={distilled_obs['target_pos']}")
        if distilled_obs.get("front_object") is not None:
            context.append(f"front={distilled_obs['front_object']}")
        if distilled_obs.get("hazards"):
            context.append(f"hazards={len(distilled_obs['hazards'])}")
        return f"[TEMPLATE: {self.name}] {' | '.join(context)} | {self.reasoning_pattern}"


ALL_TEMPLATES = [
    ThoughtTemplate("Direct Navigation", "Open path to goal.", "Align with the dominant goal axis and move forward."),
    ThoughtTemplate("Obstacle Avoidance", "Hazards require detour.", "Avoid blocked front cells and rotate toward safer alternatives."),
    ThoughtTemplate("Recovery", "No clear progress.", "Break loops with controlled turns, then resume goal seeking."),
]


class MetaBuffer:
    def __init__(self):
        self.templates = {template.name: template for template in ALL_TEMPLATES}

    def retrieve(self, distilled_obs):
        front = distilled_obs.get("front_object")
        hazards = distilled_obs.get("hazards", [])
        if front in {"lava", "wall"} or hazards:
            return self.templates["Obstacle Avoidance"]
        return self.templates["Direct Navigation"]

    def update_stats(self, template_name, success):
        template = self.templates.get(template_name)
        if template is None:
            return
        template.usage_count += 1
        template.success_rate += (float(success) - template.success_rate) / template.usage_count


class ProblemDistiller:
    @staticmethod
    def distill(obs, env=None):
        distilled = {
            "agent_pos": None,
            "facing": None,
            "target_pos": None,
            "front_object": None,
            "hazards": [],
        }

        if env is not None and hasattr(env, "unwrapped"):
            base = env.unwrapped
            distilled["agent_pos"] = [int(base.agent_pos[0]), int(base.agent_pos[1])]
            distilled["facing"] = MinigridTextWrapper.DIR_NAMES[int(base.agent_dir)]

            fx, fy = int(base.front_pos[0]), int(base.front_pos[1])
            front = base.grid.get(fx, fy)
            distilled["front_object"] = front.type if front is not None else "empty"

            for x in range(base.grid.width):
                for y in range(base.grid.height):
                    cell = base.grid.get(x, y)
                    if cell is None:
                        continue
                    if cell.type == "goal":
                        distilled["target_pos"] = [x, y]
                    elif cell.type == "lava":
                        distilled["hazards"].append([x, y])
            return distilled

        match = re.search(r"Agent is at \[(\d+),\s*(\d+)\] facing (\w+)", obs)
        if match:
            distilled["agent_pos"] = [int(match.group(1)), int(match.group(2))]
            distilled["facing"] = match.group(3)

        match = re.search(r"Goal is at \[(\d+),\s*(\d+)\]", obs)
        if match:
            distilled["target_pos"] = [int(match.group(1)), int(match.group(2))]

        match = re.search(r"The cell directly in front of you contains ([^.]+)\.", obs)
        if match:
            distilled["front_object"] = match.group(1).strip()

        return distilled


class BoTAgent:
    DIRS = ["right", "down", "left", "up"]
    VEC = {"right": (1, 0), "down": (0, 1), "left": (-1, 0), "up": (0, -1)}

    def __init__(self, h_size=6, branch_depth=3):
        self.buffer = MetaBuffer()
        self.h_size = h_size
        self.branch_depth = branch_depth
        self.memory = []
        self._last_template = None
        self._recent_states = []

    def _dir_index(self, facing):
        return self.DIRS.index(facing) if facing in self.DIRS else 0

    def _turn(self, facing, action):
        idx = self._dir_index(facing)
        if action == "turn_left":
            return self.DIRS[(idx - 1) % 4]
        if action == "turn_right":
            return self.DIRS[(idx + 1) % 4]
        return facing

    def _blocked(self, env, x, y):
        if env is None:
            return False
        base = env.unwrapped
        if x < 0 or y < 0 or x >= base.grid.width or y >= base.grid.height:
            return True
        cell = base.grid.get(x, y)
        if cell is None:
            return False
        return getattr(cell, "type", None) in {"wall", "lava"}

    def _simulate_step(self, state, action, env=None):
        x, y, facing = state
        if action in {"turn_left", "turn_right"}:
            return (x, y, self._turn(facing, action)), False

        dx, dy = self.VEC[facing]
        nx, ny = x + dx, y + dy
        if self._blocked(env, nx, ny):
            return (x, y, facing), True
        return (nx, ny, facing), False

    def _candidate_sequences(self, template_name):
        if template_name == "Obstacle Avoidance":
            return [
                ["turn_left", "move_forward", "move_forward"],
                ["turn_right", "move_forward", "move_forward"],
                ["turn_left", "move_forward", "turn_right"],
                ["turn_right", "move_forward", "turn_left"],
                ["turn_left", "turn_left", "move_forward"],
            ]
        if template_name == "Recovery":
            return [
                ["turn_left", "move_forward", "turn_right"],
                ["turn_right", "move_forward", "turn_left"],
                ["turn_left", "turn_left", "move_forward"],
            ]
        return [
            ["move_forward", "move_forward", "move_forward"],
            ["turn_left", "move_forward", "move_forward"],
            ["turn_right", "move_forward", "move_forward"],
            ["move_forward", "turn_left", "move_forward"],
            ["move_forward", "turn_right", "move_forward"],
        ]

    def _score_rollout(self, states, blocks, target, visited_penalty):
        sx, sy, _ = states[-1]
        score = 0.0

        if target is not None:
            dist = abs(target[0] - sx) + abs(target[1] - sy)
            score += 30.0 - (4.0 * dist)

        score -= 6.0 * blocks
        score -= visited_penalty
        return score

    def _choose_action(self, distilled, template, env=None):
        if distilled["agent_pos"] is None or distilled["facing"] is None:
            return "turn_right", "fallback:no_state"

        start = (distilled["agent_pos"][0], distilled["agent_pos"][1], distilled["facing"])
        target = tuple(distilled["target_pos"]) if distilled["target_pos"] is not None else None

        best_first = "move_forward"
        best_score = -1e9
        best_trace = None

        for seq in self._candidate_sequences(template.name):
            seq = seq[: self.branch_depth]
            state = start
            blocks = 0
            trace = [state]
            for act in seq:
                state, blocked = self._simulate_step(state, act, env)
                blocks += int(blocked)
                trace.append(state)

            repeat_penalty = 0.0
            for state_item in trace:
                repeat_penalty += self._recent_states.count(state_item) * 1.5

            score = self._score_rollout(trace, blocks, target, repeat_penalty)
            if score > best_score:
                best_score = score
                best_first = seq[0]
                best_trace = seq

        reason = f"template={template.name} best={best_trace} score={best_score:.2f}"
        return best_first, reason

    def act(self, obs, env=None):
        distilled = ProblemDistiller.distill(obs, env)
        template = self.buffer.retrieve(distilled)
        self._last_template = template.name

        if env is not None and distilled["agent_pos"] is not None and distilled["facing"] is not None:
            current_state = (distilled["agent_pos"][0], distilled["agent_pos"][1], distilled["facing"])
            if self._recent_states.count(current_state) >= 2:
                template = self.buffer.templates["Recovery"]
                self._last_template = template.name
            self._recent_states.append(current_state)
            if len(self._recent_states) > 20:
                self._recent_states = self._recent_states[-20:]

        action, reason = self._choose_action(distilled, template, env)
        response = f"{template.instantiate(distilled)} | {reason}"

        self.memory.append((obs, action))
        if len(self.memory) > self.h_size:
            self.memory = self.memory[-self.h_size:]

        return action, response, 0.0

    def learn(self, success, max_steps=100):
        _ = max_steps
        if self._last_template is not None:
            self.buffer.update_stats(self._last_template, bool(success))

    def reset(self):
        self.memory = []
        self._last_template = None
        self._recent_states = []

In [ ]:
env = MinigridTextWrapper("MiniGrid-LavaGapS6-v0")
agent = BoTAgent(h_size=4)

Loading model Qwen/Qwen2.5-7B-Instruct with Transformers (this may take a few minutes)...


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


In [ ]:
print("\n--- Evaluating on MiniGrid-Empty-8x8-v0 ---")
evaluate_agent(
    agent,
    env_name="MiniGrid-Empty-8x8-v0",
    num_episodes=10,
    max_steps_per_episode=100,
    save_gifs=False,
    seed_base=42,
 )

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Evaluating on MiniGrid-Empty-8x8-v0 ---
Evaluating MiniGrid-Empty-8x8-v0 for 10 episodes


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Episode 1/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_1.gif
Episode 2/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_2.gif
Episode 3/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_3.gif
Episode 4/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_4.gif
Episode 5/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_5.gif
Episode 6/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_6.gif
Episode 7/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_7.gif
Episode 8/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_8.gif
Episode 9/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_9.gif
Episode 10/10 | Success: True | Steps: 13 | GIF saved: episode_gifs/episode_10.gif

🏆 Final Metrics
Success Rate: 100.00%
Avg Steps (successful episodes): 13.00
Avg Inference Time per step: 2.7666 sec


{'success_count': 10,
 'total_steps_success': 130,
 'total_inference_time': 359.6624290943146,
 'total_tokens': 0,
 'total_actions': 130}

In [ ]:
print("\n--- Evaluating on MiniGrid-LavaGapS6-v0 ---")
evaluate_agent(
    agent,
    env_name="MiniGrid-LavaGapS6-v0",
    num_episodes=10,
    max_steps_per_episode=100,
    save_gifs=False,
    seed_base=42,
 )


--- Evaluating on MiniGrid-LavaGapS6-v0 ---
Evaluating MiniGrid-LavaGapS6-v0 for 10 episodes
Episode 1/10 | Success: True | Steps: 9 | GIF saved: episode_gifs/episode_1.gif
Episode 2/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_2.gif
Episode 3/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_3.gif
Episode 4/10 | Success: True | Steps: 9 | GIF saved: episode_gifs/episode_4.gif
Episode 5/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_5.gif
Episode 6/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_6.gif
Episode 7/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_7.gif
Episode 8/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_8.gif
Episode 9/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_9.gif
Episode 10/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_10.gif

🏆 Final Metrics
Success Rate: 20.00%
Avg Steps (successful episodes): 9.00
Avg 

{'success_count': 2,
 'total_steps_success': 18,
 'total_inference_time': 2287.9224231243134,
 'total_tokens': 0,
 'total_actions': 818}

In [ ]:
print("\n--- Evaluating on MiniGrid-LavaCrossingS9N2-v0 ---")
evaluate_agent(
    agent,
    env_name="MiniGrid-LavaCrossingS9N2-v0",
    num_episodes=10,
    max_steps_per_episode=100,
    save_gifs=False,
    seed_base=42,
 )


--- Evaluating on MiniGrid-LavaCrossingS9N2-v0 ---
Evaluating MiniGrid-LavaCrossingS9N2-v0 for 10 episodes
Episode 1/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_1.gif
Episode 2/10 | Success: True | Steps: 15 | GIF saved: episode_gifs/episode_2.gif
Episode 3/10 | Success: False | Steps: 40 | GIF saved: episode_gifs/episode_3.gif
Episode 4/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_4.gif
Episode 5/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_5.gif
Episode 6/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_6.gif
Episode 7/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_7.gif
Episode 8/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_8.gif
Episode 9/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_9.gif
Episode 10/10 | Success: False | Steps: 100 | GIF saved: episode_gifs/episode_10.gif

🏆 Final Metrics
Success Rate: 10.00%
Avg Steps (successful epi

{'success_count': 1,
 'total_steps_success': 15,
 'total_inference_time': 2920.4363052845,
 'total_tokens': 0,
 'total_actions': 855}